# RT-DETR-L feasibility test on free Colab (T4)

**Question this notebook answers:** can RT-DETR-L be fine-tuned on the CQUniversity fire/smoke dataset (8,826 train / 1,088 val / 1,107 test images, 640x640) on the free Colab T4 within a realistic time budget?

The notebook downloads the dataset by itself (official CQUniversity file on Figshare, ~545 MB), verifies its MD5, extracts it, fixes `data.yaml`, and runs a short test (**2 epochs on ~10 % of the training images, full validation split**). It measures time, GPU memory and errors, extrapolates to a full run, and saves a report to Google Drive. **Google Drive is used only for the results and one checkpoint, never for the dataset.**

**It does not measure accuracy.** After two short epochs on ~900 images an RT-DETR model scores close to zero. That is normal and says nothing about the final model.

**How to run**
1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > **Run all**. Nothing needs editing. Accept the Google Drive permission pop-up (it is only used to save the report and one checkpoint).
3. Rough guess: 10-30 minutes in total (download, unzip, two short epochs, three validation passes).
4. Read the **REPORT** printed by the last code cell. It is also saved to Drive.

If a cell says pip changed PyTorch, use Runtime > Restart session, then Run all again.

In [1]:
# ======================= SETTINGS (nothing here needs to be edited) =======================
# --- dataset: downloaded automatically, NOT taken from Google Drive
DATASET_URL    = "https://ndownloader.figshare.com/files/53486522"   # official CQUniversity record, file MyData_Fire.zip
ZIP_PATH       = "/content/MyData_Fire.zip"
EXTRACT_DIR    = "/content/MyData_Fire"
EXPECTED_MD5   = "fd9ed42cd67fd5f7c016ca127985e245"                  # MD5 published on the Figshare record
EXPECTED_BYTES = 544994383
EXPECTED_SPLITS = {"train": 8826, "val": 1088, "test": 1107}         # sanity check only

# --- Google Drive: results and one checkpoint only
RESULTS_DIR = "/content/drive/MyDrive/rtdetr_feasibility"
SAVE_CHECKPOINT_TO_DRIVE = True   # copies last.pt to Drive and times the copy (useful for planning per-epoch checkpoints)

# --- the feasibility test
MODEL        = "rtdetr-l.pt"
FRACTION     = 0.10            # share of the TRAINING images used for the test (random sample)
EPOCHS       = 2               # epoch 1 includes warm-up; the time estimate is based on the LAST epoch (epoch 2)
IMGSZ        = 640
BATCH_SIZES  = [16, 8]         # tried in this order; 8 is used automatically if 16 runs out of CUDA memory
WORKERS      = 2               # free Colab has only 2 CPU cores
CACHE        = False           # False or "disk"  ("ram" will not fit 8.8k images in free Colab RAM)
AMP          = True            # mixed precision. If you see NaN losses, set False and re-run
SEED         = 0

PLANNED_EPOCHS    = 50         # the epoch budget you want to extrapolate to
HOURS_COMFORTABLE = 4.0        # estimated total up to this many hours  -> KEEP
HOURS_MAX         = 8.0        # estimated total above this many hours -> REPLACE

INSTALL_ULTRALYTICS = True
ULTRALYTICS_SPEC    = "ultralytics"   # latest; or pin, e.g. "ultralytics==8.3.0"
REQUIRE_GPU         = True

## Step 1 - Check the GPU

In [2]:
import os, sys, time, json, math, gc, shutil, subprocess, platform, traceback, datetime, random, threading
from pathlib import Path

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return ""

print("Python:", platform.python_version())
print(sh("nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv") or "nvidia-smi not available")

import torch
HAS_GPU = torch.cuda.is_available()
GPU_NAME, GPU_VRAM_MIB = None, None
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME, GPU_VRAM_MIB = props.name, props.total_memory / 2**20
    print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_MIB/1024:.1f} GiB | torch {torch.__version__} | CUDA {torch.version.cuda}")
    if "T4" not in GPU_NAME:
        print("WARNING: this is not a T4. Timings will NOT represent the free-tier T4 you plan to use.")
else:
    msg = "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then Runtime > Restart session and Run all."
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

import psutil
_disk_root = "/content" if os.path.exists("/content") else "/"
CPU_CORES = os.cpu_count()
RAM_GIB = psutil.virtual_memory().total / 2**30
print(f"CPU cores: {CPU_CORES} | RAM: {RAM_GIB:.1f} GiB | free disk: {shutil.disk_usage(_disk_root).free/2**30:.0f} GiB")

Python: 3.13.15
name, memory.total [MiB], memory.used [MiB], driver_version
Tesla T4, 15360 MiB, 0 MiB, 580.82.07
GPU: Tesla T4 | VRAM: 14.6 GiB | torch 2.11.0+cu130 | CUDA 13.0
CPU cores: 2 | RAM: 12.7 GiB | free disk: 70 GiB


## Step 2 - Install Ultralytics

In [3]:
if INSTALL_ULTRALYTICS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", ULTRALYTICS_SPEC], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError("pip install failed - see the messages above.")

try:
    from importlib.metadata import version as _pkg_version
    if _pkg_version("torch").split("+")[0] != torch.__version__.split("+")[0]:
        print("NOTE: pip changed the PyTorch version. Use Runtime > Restart session, then Run all again.")
except Exception:
    pass

os.environ.setdefault("WANDB_MODE", "disabled")
import ultralytics
from ultralytics import RTDETR
from ultralytics.utils import LOGGER
ULTRALYTICS_VERSION = ultralytics.__version__
print("Ultralytics", ULTRALYTICS_VERSION, "| torch", torch.__version__)
ultralytics.checks()

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.4/112.6 GB disk)


## Step 3 - Mount Google Drive (for the results only) and set up the output folder

In [4]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except Exception as e:
    ON_COLAB = False
    print(f"Google Drive was not mounted ({type(e).__name__}). Results will be saved to a local folder instead.")

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(RESULTS_DIR) / f"run_{RUN_ID}"
try:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / ".write_test").write_text("ok"); (OUT_DIR / ".write_test").unlink()
except Exception as e:
    OUT_DIR = Path(_disk_root) / "rtdetr_feasibility_local" / f"run_{RUN_ID}"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"WARNING: cannot write to RESULTS_DIR ({e}). Using {OUT_DIR}. Download or copy it before the session ends.")

import logging
LOG_PATH = OUT_DIR / "ultralytics_train.log"
if "_fh" in globals():
    LOGGER.removeHandler(_fh)          # avoid duplicate log lines if this cell is run twice
_fh = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
LOGGER.addHandler(_fh)
print("Results will be saved to:", OUT_DIR)

Google Drive was not mounted (MessageError). Results will be saved to a local folder instead.
Results will be saved to: /content/drive/MyDrive/rtdetr_feasibility/run_20261004_183130


## Step 4 - Download and verify the dataset, extract it, fix `data.yaml`, build the 10 % training subset

* Downloads `MyData_Fire.zip` from the official Figshare link to `/content` (local Colab disk) and checks its **MD5**. If a verified copy is already there, the download is skipped.
* Extracts to `/content/MyData_Fire`.
* The Roboflow `data.yaml` uses `../train/images`-style paths that Ultralytics cannot resolve. The original is kept as `data.original.yaml`, and `data.yaml` is rewritten with absolute paths (reusable for the real training later).
* The test uses a separate YAML with a random 10 % list of training images (`train_subset.txt`) and the **full** validation split.

If the download fails (rare), open the URL in your browser, upload the file to `/content/MyData_Fire.zip` through the Files panel on the left, and re-run this cell. It will verify the MD5 of whatever is there.

In [5]:
import zipfile, yaml, hashlib, collections, requests
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ZIP_P, EXTRACT_P = Path(ZIP_PATH), Path(EXTRACT_DIR)
WORK = ZIP_P.parent / "feasibility_work"
RUNS_DIR = ZIP_P.parent / "runs_feasibility"
WORK.mkdir(parents=True, exist_ok=True)
DOWNLOAD_INFO = {"url": DATASET_URL, "expected_md5": EXPECTED_MD5}

def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def download_verified(url, dest, expected_md5, expected_bytes=None, attempts=3):
    dest = Path(dest)
    if dest.exists():
        print(f"{dest} already exists - verifying MD5 ...")
        got = md5_of(dest)
        if got == expected_md5:
            print("MD5 OK (existing file):", got)
            DOWNLOAD_INFO.update(reused_existing_file=True, md5=got, md5_ok=True, bytes=dest.stat().st_size)
            return
        print(f"Existing file has MD5 {got}, expected {expected_md5} -> deleting it and downloading again")
        dest.unlink()
    part = dest.with_name(dest.name + ".part")
    last_err = None
    for k in range(1, attempts + 1):
        try:
            print(f"Download attempt {k}/{attempts}: {url}")
            t0 = time.perf_counter(); h = hashlib.md5(); n = 0; next_report = 0
            with requests.get(url, stream=True, timeout=(20, 60), headers={"User-Agent": "Mozilla/5.0"}) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0)) or expected_bytes
                with open(part, "wb") as out:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        if not chunk:
                            continue
                        out.write(chunk); h.update(chunk); n += len(chunk)
                        if total and n >= next_report:
                            print(f"  {n/2**20:6.0f} / {total/2**20:.0f} MiB ({100*n/total:3.0f} %)  {n/2**20/max(time.perf_counter()-t0, 1e-6):5.1f} MiB/s")
                            next_report += total * 0.1
            secs = time.perf_counter() - t0
            md5 = h.hexdigest()
            if expected_bytes and n != expected_bytes:
                raise IOError(f"received {n} bytes, expected {expected_bytes}")
            if md5 != expected_md5:
                raise ValueError(f"MD5 mismatch: got {md5}, expected {expected_md5}")
            part.replace(dest)
            print(f"Downloaded {n/2**20:.0f} MiB in {secs:.0f} s. MD5 OK: {md5}")
            DOWNLOAD_INFO.update(reused_existing_file=False, md5=md5, md5_ok=True, bytes=n, seconds=round(secs, 1))
            return
        except Exception as e:
            last_err = e
            print(f"  attempt {k} failed: {type(e).__name__}: {str(e)[:200]}")
            part.unlink(missing_ok=True)
            time.sleep(5)
    raise RuntimeError(f"Could not download a valid copy after {attempts} attempts (last error: {last_err}).\n"
                       f"Open {url} in your browser, upload the file to {dest} using the Files panel, and re-run this cell.")

download_verified(DATASET_URL, ZIP_P, EXPECTED_MD5, EXPECTED_BYTES)

# ---- extract to /content/MyData_Fire (skipped if a complete extraction is already there)
marker = EXTRACT_P / ".extract_complete"
if marker.exists():
    print("Already extracted:", EXTRACT_P)
else:
    if EXTRACT_P.exists():
        shutil.rmtree(EXTRACT_P)                       # an earlier extraction was interrupted
    t0 = time.perf_counter(); print(f"Extracting to {EXTRACT_P} ...")
    with zipfile.ZipFile(ZIP_P) as z:
        z.extractall(EXTRACT_P)
    marker.write_text("ok")
    print(f"Extracted in {time.perf_counter()-t0:.0f} s")

# ---- locate data.yaml, keep the original, rewrite it with working paths
yamls = sorted(EXTRACT_P.rglob("data.yaml"), key=lambda p: len(p.parts))
ds_root = yamls[0].parent if yamls else EXTRACT_P
orig_yaml = ds_root / "data.original.yaml"
if yamls and not orig_yaml.exists():
    shutil.copy(yamls[0], orig_yaml)
orig_cfg = yaml.safe_load(open(orig_yaml)) if orig_yaml.exists() else {}
if not yamls:
    print("WARNING: no data.yaml found in the archive; building one from the folder layout.")
print("\nDataset root:", ds_root)
print("BEFORE (original data.yaml):", {k: orig_cfg[k] for k in ("train", "val", "test", "nc", "names") if k in orig_cfg})

SPLIT_NAMES = {"train": ["train"], "val": ["valid", "val", "validation"], "test": ["test"]}
split_dirs = {}
for key, names_ in SPLIT_NAMES.items():
    found = None
    for n_ in names_:
        if (ds_root / n_ / "images").is_dir():
            found = ds_root / n_ / "images"; break
    if found is None:
        for p in ds_root.rglob("images"):
            if p.is_dir() and p.parent.name.lower() in names_:
                found = p; break
    split_dirs[key] = found
if split_dirs["train"] is None or split_dirs["val"] is None:
    raise RuntimeError(f"Could not find train/ and valid/ image folders under {ds_root}. Found: {split_dirs}")

names = orig_cfg.get("names", ["fire", "smoke"])
_n = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
if _n != ["fire", "smoke"]:
    print("WARNING: class names are", names, "- expected fire, smoke.")

fixed = dict(orig_cfg)                                  # keeps nc, names and the roboflow block
fixed.update({"path": str(ds_root), "train": str(split_dirs["train"]), "val": str(split_dirs["val"]), "names": names, "nc": len(_n)})
if split_dirs["test"] is not None:
    fixed["test"] = str(split_dirs["test"])
else:
    fixed.pop("test", None)
yaml.safe_dump(fixed, open(ds_root / "data.yaml", "w"), sort_keys=False)
print("AFTER  (fixed data.yaml)   :", {k: fixed[k] for k in ("path", "train", "val", "test", "nc", "names") if k in fixed})

def list_images(d):
    return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXT)

DATASET_STATS = {}
print("\nsplit  images  labels   class0   class1   expected  ok?")
for key, d in split_dirs.items():
    if d is None:
        continue
    imgs = list_images(d)
    lbl_dir = d.parent / "labels"
    lbl_files = sorted(lbl_dir.glob("*.txt")) if lbl_dir.is_dir() else []
    inst = collections.Counter()
    for lf in lbl_files:
        for line in lf.read_text().splitlines():
            if line.strip():
                inst[line.split()[0]] += 1
    DATASET_STATS[key] = {"images": len(imgs), "labels": len(lbl_files), "instances": {k_: inst[k_] for k_ in sorted(inst)}}
    exp = EXPECTED_SPLITS.get(key)
    ok = "yes" if exp == len(imgs) else f"NO (expected {exp})"
    print(f"{key:<6} {len(imgs):>6} {len(lbl_files):>7} {inst['0']:>8} {inst['1']:>8} {exp!s:>9}  {ok}")

train_imgs = list_images(split_dirs["train"])
sample_sizes = collections.Counter(Image.open(p).size for p in random.Random(0).sample(train_imgs, min(200, len(train_imgs))))
print("Image sizes in a 200-image sample:", dict(sample_sizes))

# ---- feasibility YAML: random subset of the training images, complete validation split
k = max(1, round(FRACTION * len(train_imgs)))
subset = sorted(random.Random(SEED).sample(train_imgs, k))
subset_txt = WORK / "train_subset.txt"
subset_txt.write_text("\n".join(str(p.resolve()) for p in subset) + "\n")
cfg = {"path": str(ds_root), "train": str(subset_txt), "val": str(split_dirs["val"]), "names": names, "nc": len(_n)}
if split_dirs["test"] is not None:
    cfg["test"] = str(split_dirs["test"])
YAML_PATH = WORK / "fire_feasibility.yaml"
yaml.safe_dump(cfg, open(YAML_PATH, "w"), sort_keys=False)
for src_file in (YAML_PATH, subset_txt, ds_root / "data.yaml", orig_yaml):
    if src_file.exists():
        shutil.copy(src_file, OUT_DIR / (src_file.name if src_file.name != "data.yaml" else "data_fixed.yaml"))

print(f"\nTraining subset: {k} of {len(train_imgs)} training images ({100*k/len(train_imgs):.1f} %), validation: {DATASET_STATS['val']['images']} images")
print("Feasibility YAML:", YAML_PATH); print(open(YAML_PATH).read())

Download attempt 1/3: https://ndownloader.figshare.com/files/53486522
       1 / 520 MiB (  0 %)    0.5 MiB/s
      52 / 520 MiB ( 10 %)   11.5 MiB/s
     104 / 520 MiB ( 20 %)   14.8 MiB/s
     156 / 520 MiB ( 30 %)   16.4 MiB/s
     208 / 520 MiB ( 40 %)   17.4 MiB/s
     260 / 520 MiB ( 50 %)   18.0 MiB/s
     312 / 520 MiB ( 60 %)   18.5 MiB/s
     364 / 520 MiB ( 70 %)   18.9 MiB/s
     416 / 520 MiB ( 80 %)   19.2 MiB/s
     468 / 520 MiB ( 90 %)   19.4 MiB/s
     520 / 520 MiB (100 %)   19.5 MiB/s
Downloaded 520 MiB in 27 s. MD5 OK: fd9ed42cd67fd5f7c016ca127985e245
Extracting to /content/MyData_Fire ...
Extracted in 3 s

Dataset root: /content/MyData_Fire
BEFORE (original data.yaml): {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 2, 'names': ['fire', 'smoke']}
AFTER  (fixed data.yaml)   : {'path': '/content/MyData_Fire', 'train': '/content/MyData_Fire/train/images', 'val': '/content/MyData_Fire/valid/images', 'test': '/content/MyData_Fire/

## Step 5 - Test harness: monitoring, timing callbacks and automatic out-of-memory retry

Nothing is trained in this cell; it only defines the helpers.

In [6]:
class GpuMonitor(threading.Thread):
    # Polls nvidia-smi every 2 s: memory used (whole GPU, MiB) and GPU utilisation (%).
    def __init__(self, interval=2.0):
        super().__init__(daemon=True)
        self.interval = interval
        self.halt = threading.Event()
        self.samples = []          # (perf_counter, mem_used_MiB, util_percent)
    def run(self):
        while not self.halt.is_set():
            out = sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits")
            try:
                mem, util = [float(x) for x in out.splitlines()[0].split(",")]
                self.samples.append((time.perf_counter(), mem, util))
            except Exception:
                pass
            self.halt.wait(self.interval)
    def stop(self):
        self.halt.set()

EP = {}                  # per-epoch timing records
LOSS_TRACE = []          # summed loss after every batch
NONFINITE = {"count": 0} # batches whose loss was NaN/inf
INFO = {}                # facts read from the trainer (actual batch size, batches per epoch, ...)

def _rec(trainer):
    return EP.setdefault(int(getattr(trainer, "epoch", 0)), {"batch_t": []})

def _loss_values(trainer):
    try:
        li = getattr(trainer, "loss_items", None)
        if li is None:
            return []
        if isinstance(li, dict):
            vals = list(li.values())
        elif hasattr(li, "detach"):
            vals = li.detach().float().cpu().flatten().tolist()
        else:
            vals = list(li)
        return [float(v) for v in vals]
    except Exception:
        return []

def cb_epoch_start(trainer):
    _rec(trainer)["t_start"] = time.perf_counter()
    try:
        INFO["batch_size_actual"] = int(trainer.batch_size)
        INFO["batches_per_epoch"] = len(trainer.train_loader)
        INFO["train_images_used"] = len(trainer.train_loader.dataset)
    except Exception:
        pass

def cb_batch_end(trainer):
    _rec(trainer)["batch_t"].append(time.perf_counter())
    try:
        bs_now = int(trainer.batch_size)
        if INFO.get("batch_size_actual") and bs_now != INFO["batch_size_actual"]:
            INFO["batch_size_changed_from"] = INFO["batch_size_actual"]
            INFO["batch_size_actual"] = bs_now
    except Exception:
        pass
    vals = _loss_values(trainer)
    if vals:
        if not all(math.isfinite(v) for v in vals):
            NONFINITE["count"] += 1
        LOSS_TRACE.append(sum(v for v in vals if math.isfinite(v)))

def cb_train_epoch_end(trainer):      # fires after the last training batch, BEFORE validation
    _rec(trainer)["t_train_end"] = time.perf_counter()

def cb_fit_epoch_end(trainer):        # fires after validation
    _rec(trainer)["t_fit_end"] = time.perf_counter()

def cb_train_end(trainer):            # fires after the final evaluation of best.pt
    EP.setdefault("final", {})["t_train_end_cb"] = time.perf_counter()

def is_oom(exc):
    return isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower()

def train_once(batch):
    EP.clear(); LOSS_TRACE.clear(); NONFINITE["count"] = 0; INFO.clear()
    model = RTDETR(MODEL)
    for ev, fn in [("on_train_epoch_start", cb_epoch_start), ("on_train_batch_end", cb_batch_end),
                   ("on_train_epoch_end", cb_train_epoch_end), ("on_fit_epoch_end", cb_fit_epoch_end),
                   ("on_train_end", cb_train_end)]:
        model.add_callback(ev, fn)
    model.train(data=str(YAML_PATH), epochs=EPOCHS, imgsz=IMGSZ, batch=batch, workers=WORKERS,
                device=0 if HAS_GPU else "cpu", amp=AMP, seed=SEED, deterministic=False,
                cache=CACHE, plots=False, val=True,
                project=str(RUNS_DIR), name=f"rtdetr_l_bs{batch}", exist_ok=True)
    return model

def free_gpu():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()
    time.sleep(2)
print("Harness ready.")

Harness ready.


## Step 6 - Run the feasibility test

Watch the progress bar: the **it/s** value and the `GPU_mem` column are the most useful live numbers. If CUDA runs out of memory the notebook retries with the next batch size in `BATCH_SIZES`.

In [7]:
attempts, model, fatal_tb, success_batch = [], None, None, None
monitor = GpuMonitor(); monitor.start()
wall0 = time.perf_counter()

for b in BATCH_SIZES:
    print(f"\n==================== Attempt: batch size {b} ====================")
    rec, oom, t0 = {"batch": b}, False, time.perf_counter()
    try:
        if HAS_GPU:
            torch.cuda.reset_peak_memory_stats()
        model = train_once(b)
        rec.update(status="ok", wall_s=round(time.perf_counter() - t0, 1))
        if HAS_GPU:
            rec["peak_allocated_MiB"] = round(torch.cuda.max_memory_allocated() / 2**20)
            rec["peak_reserved_MiB"] = round(torch.cuda.max_memory_reserved() / 2**20)
        success_batch = b
    except Exception as e:
        oom = is_oom(e)
        rec.update(status="OOM" if oom else "error", error=str(e)[:400], wall_s=round(time.perf_counter() - t0, 1))
        if not oom:
            fatal_tb = traceback.format_exc()
        print("!! attempt failed:", rec["status"], "-", str(e)[:300])
    attempts.append(rec)
    if success_batch is None:
        model = None
    free_gpu()                      # the exception (and its GPU tensors) is released here
    if success_batch is not None or not oom:
        break
    print(f"CUDA out of memory at batch {b} -> trying the next batch size ...")

monitor.stop()
WALL_TOTAL_S = time.perf_counter() - wall0
print(f"\nDone. Wall-clock for all attempts: {WALL_TOTAL_S/60:.1f} min | batch size that worked: {success_batch}")
if success_batch is None and fatal_tb is None:
    print("\nCUDA ran out of memory even at the smallest batch size in BATCH_SIZES.\n"
          "Try BATCH_SIZES = [2, 1] and/or IMGSZ = 512, but treat this as a strong sign that RT-DETR-L is too heavy for the free T4.")
if fatal_tb:
    print("\nA non-memory error stopped the test. Full traceback:\n", fatal_tb)


==================== Attempt: batch size 16 ====================
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/feasibility_work/fire_feasibility.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.

## Step 7 - Report, extrapolation and verdict (also saved to Google Drive)

In [8]:
import csv, statistics

def read_csv_rows(path):
    rows = []
    try:
        with open(path, newline="") as f:
            for r in csv.DictReader(f):
                rows.append({(k or "").strip(): (v.strip() if isinstance(v, str) else v) for k, v in r.items()})
    except Exception:
        pass
    return rows

def num(x):
    try:
        return float(x)
    except Exception:
        return None

# ---------- locate the run folder and the Ultralytics results.csv
run_dir = None
if model is not None and getattr(model, "trainer", None) is not None:
    run_dir = Path(model.trainer.save_dir)
elif attempts:
    run_dir = RUNS_DIR / f"rtdetr_l_bs{attempts[-1]['batch']}"
rows = read_csv_rows(run_dir / "results.csv") if run_dir else []
last = rows[-1] if rows else {}
csv_nonfinite = [k for r in rows for k, v in r.items() if k != "epoch" and num(v) is not None and not math.isfinite(num(v))]

# ---------- timing from the callbacks: the estimate is based on the LAST completed epoch (epoch 2 when EPOCHS = 2)
def epoch_timing(e):
    r = EP[e]; bt = r["batch_t"]
    diffs = [b - a for a, b in zip(bt, bt[1:])]
    skip = min(10, len(diffs) // 4)               # ignore the first batches (CUDA/cuDNN warm-up, worker start-up)
    steady = diffs[skip:] or diffs
    if not steady:
        return None
    return {
        "epoch": e + 1, "batches": len(bt), "batches_timed": len(steady),
        "s_per_batch_mean": statistics.mean(steady),
        "s_per_batch_median": statistics.median(steady),
        "s_per_batch_p95": sorted(steady)[int(0.95 * (len(steady) - 1))],
        "train_epoch_s": r["t_train_end"] - r["t_start"],
        "val_s": (r["t_fit_end"] - r["t_train_end"]) if r.get("t_fit_end") else None,     # validation + metrics + checkpoint save
        "epoch_total_s": (r["t_fit_end"] - r["t_start"]) if r.get("t_fit_end") else None,
    }

ep_ids = sorted(k for k, v in EP.items() if isinstance(k, int) and v.get("t_train_end"))
pe = {e: t for e, t in ((e, epoch_timing(e)) for e in ep_ids) if t}
per_epoch = [pe[e] for e in sorted(pe)]
ref_id = max(pe) if pe else None
ref = EP[ref_id] if pe else None
timing = {}
if ref:
    timing = dict(pe[ref_id]); timing["reference_epoch"] = ref_id + 1
final_t = EP.get("final", {}).get("t_train_end_cb")
if final_t and ref and ref.get("t_fit_end"):
    timing["final_eval_and_save_s"] = final_t - ref["t_fit_end"]

# ---------- GPU monitor window = the training part of the reference epoch
gpu = {"peak_mem_used_MiB_nvidia_smi": None, "mean_util_pct_during_training": None}
if monitor.samples:
    gpu["peak_mem_used_MiB_nvidia_smi"] = max(m for _, m, _ in monitor.samples)
    if ref:
        win = [u for t, _, u in monitor.samples if ref["t_start"] <= t <= ref["t_train_end"]]
        if win:
            gpu["mean_util_pct_during_training"] = statistics.mean(win)
ok_attempt = next((a for a in attempts if a["status"] == "ok"), {})

# ---------- copy one checkpoint to Drive and time it (tells you what per-epoch checkpoints cost)
CKPT_COPY = {}
if SAVE_CHECKPOINT_TO_DRIVE and run_dir and (run_dir / "weights" / "last.pt").exists():
    try:
        dst = OUT_DIR / "weights"; dst.mkdir(exist_ok=True)
        src_ck = run_dir / "weights" / "last.pt"
        t0 = time.perf_counter(); shutil.copy(src_ck, dst / "last.pt")
        CKPT_COPY = {"file": "last.pt", "MiB": round(src_ck.stat().st_size / 2**20, 1),
                     "seconds": round(time.perf_counter() - t0, 1), "destination": str(dst)}
    except Exception as e:
        CKPT_COPY = {"error": f"{type(e).__name__}: {e}"}

# ---------- extrapolation to a full run (based on the reference epoch = epoch 2)
bs_used = INFO.get("batch_size_actual", success_batch)
EST_EPOCHS = sorted({30, 50, PLANNED_EPOCHS})
extra = {}
if timing and bs_used:
    n_train_full = DATASET_STATS["train"]["images"]
    batches_full = math.ceil(n_train_full / bs_used)

    def estimate(t):                               # t = timing record of one epoch
        est_train = t["s_per_batch_mean"] * batches_full
        val_s = t.get("val_s") or 0.0
        est_epoch = est_train + val_s + 5          # + a few seconds of per-epoch overhead
        hours = {e: (e * est_epoch + val_s + 120) / 3600 for e in EST_EPOCHS}   # + final evaluation + start-up
        return est_train, val_s, est_epoch, hours

    est_train, val_s, est_epoch, hours = estimate(timing)
    extra = {"batch_size_used": bs_used, "batches_per_full_epoch": batches_full, "basis_epoch": timing["reference_epoch"],
             "est_full_train_epoch_min": est_train / 60, "est_full_validation_and_save_min": val_s / 60,
             "est_full_epoch_incl_val_min": est_epoch / 60,
             "est_total_h_by_epochs": {str(e): h for e, h in hours.items()},
             "est_total_h_for_planned_epochs": hours[PLANNED_EPOCHS], "planned_epochs": PLANNED_EPOCHS}
    if len(per_epoch) > 1 and per_epoch[0]["epoch"] == 1 and timing["reference_epoch"] != 1:
        extra["for_comparison_est_total_h_if_epoch1_were_used"] = {str(e): h for e, h in estimate(per_epoch[0])[3].items()}

# ---------- loss trend and NaN checks
loss_first = statistics.mean(LOSS_TRACE[:10]) if len(LOSS_TRACE) >= 20 else None
loss_last = statistics.mean(LOSS_TRACE[-10:]) if len(LOSS_TRACE) >= 20 else None
nonfinite_total = NONFINITE["count"] + len(csv_nonfinite)

# ---------- verdict
reasons, nxt = [], []
if fatal_tb:
    code_v = "INVESTIGATE"; head = "A non-memory error stopped the test. This is not evidence against RT-DETR-L yet."
    nxt.append("Read the traceback above / in the log. Fix the error (or send it to me) and re-run before deciding.")
elif success_batch is None:
    code_v = "REPLACE"; head = "CUDA ran out of memory at every batch size tried. RT-DETR-L does not fit the free T4 at 640 px."
    nxt.append("Replace RT-DETR-L with a lighter model (see the interpretation guide).")
elif nonfinite_total > 0:
    code_v = "INVESTIGATE"; head = f"NaN/inf appeared in {nonfinite_total} loss value(s). Training was unstable."
    nxt.append("Re-run with AMP = False (slower, more memory). If NaNs persist at a low batch size, replace the model.")
else:
    est_h = extra.get("est_total_h_for_planned_epochs")
    util = gpu["mean_util_pct_during_training"]
    if est_h is None:
        code_v = "INCOMPLETE"; head = "Training ran but timing could not be measured (see the log)."
    elif est_h <= HOURS_COMFORTABLE:
        code_v = "KEEP"; head = f"RT-DETR-L looks practical: about {est_h:.1f} h for {PLANNED_EPOCHS} epochs."
    elif est_h <= HOURS_MAX:
        code_v = "KEEP, WITH A REDUCED BUDGET"; head = f"RT-DETR-L is possible but slow: about {est_h:.1f} h for {PLANNED_EPOCHS} epochs."
        nxt.append("Use a smaller common epoch budget for all three models (e.g. 30), and/or split training over several sessions with checkpoints on Drive.")
    else:
        code_v = "REPLACE"; head = f"RT-DETR-L is too slow for the free T4: about {est_h:.1f} h for {PLANNED_EPOCHS} epochs."
        nxt.append("Replace it with a lighter model (see the interpretation guide).")
    if success_batch <= 2:
        reasons.append(f"Only batch size {success_batch} fit. RT-DETR-L uses BatchNorm, whose statistics become noisy at such small batches, and small batches are slower.")
        if code_v == "KEEP":
            code_v = "KEEP, BUT LOOK AT THE SMALL BATCH SIZE"
    if INFO.get("batch_size_changed_from"):
        reasons.append(f"Ultralytics itself reduced the batch size from {INFO['batch_size_changed_from']} to {INFO['batch_size_actual']} during the epoch, "
                       "so the timing mixes two batch sizes. Re-run with BATCH_SIZES set to that smaller value for a clean number.")
    if EPOCHS == 1:
        reasons.append("Timing comes from epoch 1, which includes warm-up, so the estimate is slightly pessimistic. Set EPOCHS = 2 and re-run for a cleaner number.")
    if INFO.get("batches_per_epoch") and bs_used:
        steps = INFO["batches_per_epoch"] / max(1, round(64 / bs_used)) * max(1, len(ep_ids))
        reasons.append(f"This test made only about {steps:.0f} optimizer updates in total (gradients are accumulated up to a nominal batch of 64). "
                       "'No NaN' is encouraging but not proof of stability; watch the first epochs of the real run.")
    if util is not None and util < 60:
        reasons.append(f"The GPU was only {util:.0f} % busy: the 2 CPU cores (data loading/augmentation) are the bottleneck, not the T4. "
                       "A lighter model would not fix that. Try CACHE = 'disk' and WORKERS = 2 or 3.")

# ---------- assemble and print
report = {
    "run_id": RUN_ID, "verdict": code_v, "headline": head, "reasons": reasons, "next_steps": nxt,
    "environment": {"gpu": GPU_NAME, "vram_gib": None if GPU_VRAM_MIB is None else round(GPU_VRAM_MIB / 1024, 1),
                    "cpu_cores": CPU_CORES, "ram_gib": round(RAM_GIB, 1), "ultralytics": ULTRALYTICS_VERSION,
                    "torch": torch.__version__, "python": platform.python_version()},
    "settings": {"model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "fraction": FRACTION, "batch_sizes_tried": BATCH_SIZES,
                 "workers": WORKERS, "cache": CACHE, "amp": AMP, "seed": SEED},
    "dataset": DATASET_STATS, "attempts": attempts, "trainer_info": dict(INFO), "timing": timing, "per_epoch_timing": per_epoch, "gpu": gpu,
    "memory_peak_torch_MiB": {"allocated": ok_attempt.get("peak_allocated_MiB"), "reserved": ok_attempt.get("peak_reserved_MiB")},
    "extrapolation": extra, "wall_clock_total_s": round(WALL_TOTAL_S, 1),
    "dataset_source": DOWNLOAD_INFO, "checkpoint_copy_to_drive": CKPT_COPY,
    "nan_inf": {"batches_with_nonfinite_loss": NONFINITE["count"], "nonfinite_columns_in_results_csv": csv_nonfinite},
    "validation_completed": bool(timing.get("val_s")) and bool(rows),
    "loss_trend_first10_vs_last10_batches": [loss_first, loss_last],
    "last_epoch_metrics_not_meaningful_after_tiny_test": {k: last.get(k) for k in last if "metrics/" in k or "loss" in k},
}

def fm(x, d=1):
    return "n/a" if x is None else f"{x:.{d}f}"

L = []
L.append("=" * 72); L.append("RT-DETR-L FEASIBILITY REPORT"); L.append("=" * 72)
L.append(f"VERDICT: {report['verdict']}"); L.append(report["headline"])
for r_ in reasons: L.append(f"  - {r_}")
for n_ in nxt: L.append(f"  > {n_}")
L.append("-" * 72)
L.append(f"GPU: {GPU_NAME} ({report['environment']['vram_gib']} GiB) | CPU cores: {CPU_CORES} | Ultralytics {ULTRALYTICS_VERSION}")
L.append(f"Batch size used: {bs_used} (tried: {[ (a['batch'], a['status']) for a in attempts ]})")
L.append(f"Training images in the test: {INFO.get('train_images_used')} | batches per epoch: {INFO.get('batches_per_epoch')} | validation images: {DATASET_STATS['val']['images']}")
if timing:
    ref_no = timing["reference_epoch"]
    if len(per_epoch) > 1:
        L.append(f"Timing basis: epoch {ref_no} (the last of {len(per_epoch)}). Epoch 1 contains warm-up and is shown for comparison only.")
    else:
        L.append("Timing basis: epoch 1 only. It contains warm-up, so the estimate is slightly pessimistic.")
    L.append(f"Speed in epoch {ref_no}: {fm(timing['s_per_batch_mean'],2)} s/batch mean ({fm(timing['s_per_batch_median'],2)} median, {fm(timing['s_per_batch_p95'],2)} p95) "
             f"-> {fm(bs_used / timing['s_per_batch_mean'],1)} images/s")
    L.append("Per-epoch timing of the test (seconds):")
    L.append("  epoch   batches   train   s/batch   validation+save   epoch total")
    for t_ in per_epoch:
        tag = "  <- estimate is based on this epoch" if t_["epoch"] == ref_no else ("  (warm-up)" if len(per_epoch) > 1 else "")
        L.append(f"  {t_['epoch']:>5} {t_['batches']:>9} {fm(t_['train_epoch_s'],0):>7} {fm(t_['s_per_batch_mean'],2):>9} {fm(t_.get('val_s'),0):>17} {fm(t_.get('epoch_total_s'),0):>13}{tag}")
    if timing.get("final_eval_and_save_s") is not None:
        L.append(f"Final evaluation of best.pt + save after the last epoch: {fm(timing['final_eval_and_save_s'],0)} s")
L.append(f"Total wall-clock of the test (incl. set-up): {WALL_TOTAL_S/60:.1f} min")
L.append(f"Dataset: downloaded from Figshare, MD5 verified = {DOWNLOAD_INFO.get('md5_ok')} ({DOWNLOAD_INFO.get('md5')})")
if CKPT_COPY.get("MiB") is not None:
    L.append(f"Checkpoint copy to Drive: {CKPT_COPY['file']} {CKPT_COPY['MiB']} MiB in {CKPT_COPY['seconds']} s (cost if you save a checkpoint to Drive every epoch)")
L.append(f"GPU memory peak: torch allocated {report['memory_peak_torch_MiB']['allocated']} MiB, reserved {report['memory_peak_torch_MiB']['reserved']} MiB, "
         f"nvidia-smi {fm(gpu['peak_mem_used_MiB_nvidia_smi'],0)} MiB of {fm(GPU_VRAM_MIB,0)} MiB")
L.append(f"GPU utilisation while training: {fm(gpu['mean_util_pct_during_training'],0)} %  (low = CPU/data-loading bound)")
L.append(f"Validation completed: {report['validation_completed']} | NaN/inf: {nonfinite_total} | loss first-10 vs last-10 batches: {fm(loss_first,2)} -> {fm(loss_last,2)}")
if last:
    L.append("Metrics after this tiny test (expected to be near zero, NOT a quality measure): " +
             ", ".join(f"{k.replace('metrics/','')}={last[k]}" for k in last if "mAP" in k))
if extra:
    L.append("-" * 72)
    L.append(f"EXTRAPOLATION to the full training set ({DATASET_STATS['train']['images']} images, batch {extra['batch_size_used']}, "
             f"{extra['batches_per_full_epoch']} batches/epoch), based on epoch {extra['basis_epoch']}:")
    L.append(f"  one full epoch: {fm(extra['est_full_train_epoch_min'])} min training + {fm(extra['est_full_validation_and_save_min'])} min validation/save "
             f"+ a few s overhead = {fm(extra['est_full_epoch_incl_val_min'])} min")
    for e_ in EST_EPOCHS:
        note = (f"   (thresholds: <= {HOURS_COMFORTABLE} h keep, <= {HOURS_MAX} h keep with a smaller budget, above that replace)"
                if e_ == PLANNED_EPOCHS else "")
        L.append(f"  {e_:>3} epochs: about {extra['est_total_h_by_epochs'][str(e_)]:.1f} h  (incl. final evaluation and start-up){note}")
    cmp_ = extra.get("for_comparison_est_total_h_if_epoch1_were_used")
    if cmp_:
        L.append("  for comparison, using epoch 1 (with warm-up) instead: " + ", ".join(f"{e_} epochs {cmp_[str(e_)]:.1f} h" for e_ in EST_EPOCHS))
L.append("=" * 72)
REPORT_TEXT = "\n".join(L)
print(REPORT_TEXT)

# ---------- save everything to Drive
(OUT_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
(OUT_DIR / "report.txt").write_text(REPORT_TEXT)
if run_dir and run_dir.exists():
    for name in ("results.csv", "args.yaml"):
        if (run_dir / name).exists():
            shutil.copy(run_dir / name, OUT_DIR / name)
for h_ in LOGGER.handlers:
    try: h_.flush()
    except Exception: pass
print("\nSaved to:", OUT_DIR, "->", sorted(p.name for p in OUT_DIR.iterdir()))

RT-DETR-L FEASIBILITY REPORT
VERDICT: INVESTIGATE
NaN/inf appeared in 3 loss value(s). Training was unstable.
  > Re-run with AMP = False (slower, more memory). If NaNs persist at a low batch size, replace the model.
------------------------------------------------------------------------
GPU: Tesla T4 (14.6 GiB) | CPU cores: 2 | Ultralytics 8.4.173
Batch size used: 16 (tried: [(16, 'ok')])
Training images in the test: 883 | batches per epoch: 56 | validation images: 1088
Timing basis: epoch 2 (the last of 2). Epoch 1 contains warm-up and is shown for comparison only.
Speed in epoch 2: 1.11 s/batch mean (1.11 median, 1.22 p95) -> 14.5 images/s
Per-epoch timing of the test (seconds):
  epoch   batches   train   s/batch   validation+save   epoch total
      1        56      63      1.05                30            93  (warm-up)
      2        56      62      1.11                26            88  <- estimate is based on this epoch
Final evaluation of best.pt + save after the last epoch: 

## How to read the report

| What you see | What it means | What to do |
|---|---|---|
| **KEEP** | Estimated time for the planned epochs fits the comfortable budget, no NaN, batch size 4 or more | Keep RT-DETR-L. Plan to save checkpoints to Drive and resume after disconnects |
| **KEEP, WITH A REDUCED BUDGET** | Works, but 50 epochs would take many hours | Keep it only if the common epoch budget for all three models is cut (e.g. 30), or training is split over several sessions |
| **REPLACE** | Out of memory at every batch size, or the extrapolated time is far above the budget | Swap RT-DETR-L for a lighter model |
| **INVESTIGATE** | A non-memory crash, or NaN/inf losses | Not a verdict on the model. Fix the cause first (AMP = False, library version) and re-run |
| GPU utilisation **below ~60 %** | The 2 CPU cores are the bottleneck, not the T4 | A lighter model will not help; try `CACHE = "disk"` |
| Metrics near 0 | Normal after 2 short epochs on ~900 images | Ignore them |

The three numbers that matter most: **batch size that fit**, **estimated hours for the planned epochs**, and **any NaN/inf**.